<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_9_3_Telco_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn — Classification Capstone

> **Role in this unit: the capstone — you drive.** Unlike the worked Bank Marketing example, the TASK-tagged cells here are yours to complete. Everything you need was demonstrated in notebooks 1–6 and in Bank Marketing.
**Author:** Brad Sheese

---

### Learning Objectives
By the end of this notebook you will be able to:
1. Apply the full classification workflow independently to an unfamiliar dataset.
2. Justify the choice of class imbalance strategy using the three-strategy framework from notebook 6.
3. Use out-of-fold (OOF) probabilities to select thresholds without contaminating the test set.
4. Translate threshold selection into a business cost argument.
5. Interpret nested CV results as an honest estimate of generalization performance.

---

**This is the capstone exercise for the Classification Basics module.**  
You will apply every major concept from notebooks 1–6 to a real dataset.  
Sections marked **TASK** require you to write code or a written response.


## Section 0: Setup

All imports are provided. Run this cell before anything else.

`COST_FN` and `COST_FP` represent the business costs of each error type:
- **False Negative (missed churner):** The company loses the estimated lifetime value of a customer it could have retained — estimated at **\$200**.
- **False Positive (unnecessary offer):** A retention offer is sent to a customer who was not going to churn — estimated at **\$25**.

These constants are defined globally so every section uses the same cost model.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import (
    train_test_split, cross_val_predict, cross_val_score,
    GridSearchCV, StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)

COST_FN = 200.0   # value lost from an unretained churner
COST_FP =  25.0   # cost of a retention offer sent to a non-churner


## Section 1: Load and Explore

### 1.1 Load the Data

We use the IBM Telco Customer Churn dataset.  
It contains information about 7,043 telecom customers and whether they cancelled their service (`Churn`).

**Things to notice in the output below:**
- `TotalCharges` is shown as `object` dtype even though it should be numeric. New customers have a space `' '` instead of a number; `pd.read_csv` infers the column as text as a result.
- `customerID` is a unique identifier for each row, not a predictive feature.


In [2]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print(df.shape)
df.info()


(7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    


In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 1.2 Data Cleaning — TASK

Fix the two data quality issues identified above before proceeding.

**Issue 1:** `customerID` is a unique row identifier. Keeping it would cause the model to memorize IDs instead of learning patterns. Drop it.

**Issue 2:** `TotalCharges` is stored as `object` because some new customers have a space `' '` where 0 should be. Convert it to numeric and replace the resulting `NaN` values with `0` (new customers have no accumulated charges yet).


In [4]:
# TASK: Fix the two data issues identified above.
#
# 1. Drop the customerID column (it is a unique identifier, not a feature).
#    Hint: df.drop(columns=[...], inplace=True)
#
# 2. TotalCharges is stored as object dtype because new customers have a space
#    instead of a number. Convert it to numeric and fill the resulting NaN
#    values with 0 (new customers have no total charges yet).
#    Hint: pd.to_numeric(..., errors='coerce') then .fillna(0)

# Your Code Here:


In [5]:
# Verify your cleaning: TotalCharges should now be float64, customerID gone.
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## Section 2: Target Distribution and the Accuracy Paradox — TASK

Before building any model, understand the target variable.

A **naive baseline** always predicts the majority class. Its accuracy equals the majority-class frequency.  
If a model's accuracy is only slightly above this baseline, it may not be learning anything useful.


In [6]:
# TASK: Compute and print the positive rate and the naive baseline accuracy.
#
# 1. Compute positive_rate = fraction of customers who churned (Churn == 'Yes')
# 2. Compute naive_baseline = accuracy of always predicting the majority class
#    (the larger of positive_rate and 1 - positive_rate)
# 3. Print both values with clear labels
# 4. Plot a bar chart of Churn vs. No Churn counts

# Your Code Here:


**Observation:** *(Write your answer here.)*

Is this dataset imbalanced? How does this affect the usefulness of accuracy as a metric for this problem?


## Section 3: Data Preparation

### 3.1 Encode the Target — TASK

Convert the `Churn` column from `'Yes'`/`'No'` strings to a binary integer target.


In [7]:
# TASK: Create binary target y (1 = Churn, 0 = No Churn) and feature matrix X.
# Hint: y = (df['Churn'] == 'Yes').astype(int)

# Your Code Here:


### 3.2 Identify Feature Types — TASK

The pipeline you will build needs two separate feature lists: numeric and categorical.

**Note on `SeniorCitizen`:** This column is already encoded as 0/1 (unlike the other binary columns such as `Partner` which use `'Yes'`/`'No'`).  
Treat `SeniorCitizen` as numeric.


In [8]:
# TASK: Identify numeric and categorical feature columns.
#
# Hint: Use df.select_dtypes(include=[np.number]).columns.tolist()
#       and  df.select_dtypes(include=['object']).columns.tolist()
#
# Remember to exclude 'Churn' from both lists (it is the target, not a feature).
# SeniorCitizen is 0/1 — it will land in the numeric list automatically.

# Your Code Here:

# Print the two lists so you can verify them before proceeding.
# Uncomment to verify once you have defined the two lists above:
# print('Numeric features:', numeric_features)
# Uncomment to verify once you have defined the two lists above:
# print('Categorical features:', categorical_features)


### 3.3 Build the Preprocessing Pipeline — TASK

Use a `ColumnTransformer` to apply different preprocessing to numeric and categorical columns, then wrap it in a `Pipeline` with an XGBoost classifier placeholder.

This is the same pattern used in notebook 9.1 and 9.2. You will define `xgb_model` in Section 4 and substitute it in.


In [9]:
# TASK: Build a preprocessing pipeline with two branches:
#
# 1. Numeric branch:
#    SimpleImputer(strategy='median') -> StandardScaler()
#
# 2. Categorical branch:
#    SimpleImputer(strategy='most_frequent') ->
#    OneHotEncoder(handle_unknown='ignore', drop='first')
#
# 3. Combine into a ColumnTransformer:
#    preprocessor = ColumnTransformer(transformers=[
#        ('num', numeric_pipeline, numeric_features),
#        ('cat', categorical_pipeline, categorical_features),
#    ])
#
# 4. Create a full pipeline (leave the classifier as a placeholder for now):
#    pipe_xgb = Pipeline(steps=[
#        ('preprocessor', preprocessor),
#        ('classifier', None),   # will be replaced in Section 4
#    ])

# Your Code Here:


### 3.4 Train-Test Split — TASK


In [10]:
# TASK: Split X and y into training and test sets.
#
# Use: test_size=0.2, random_state=42, stratify=y
#
# After the split, print the size and positive rate of each split
# to confirm stratification preserved the class ratio.

# Your Code Here:


## Section 4: Train XGBoost

### 4.1 Class Imbalance Decision — TASK (Written Response)

Before writing code, decide how you will handle class imbalance.  
Recall the Three Strategies framework from notebook 6:

| Approach | When it acts | What it changes | Best for |
|---|---|---|---|
| `scale_pos_weight` | Training | Counts minority examples more | Simple, moderate imbalance |
| Custom objective | Training | Scales gradient and Hessian by cost | When cost ratio ≠ class ratio |
| Threshold tuning | Prediction | Moves the decision boundary | Always useful as a second layer |

Fill in the blanks below.


**Your Decision:**

- Class ratio (negatives / positives): ___
- Cost ratio (COST_FN / COST_FP): ___
- Your choice of strategy (or strategies): ___
- Justification: *(Explain why the class ratio and cost ratio support this choice.)*


### 4.2 Train the Model — TASK

Now define `xgb_model` and plug it into the pipeline you built in Section 3.


In [11]:
# TASK: Define xgb_model and train the full pipeline.
#
# Step 1: Calculate scale_pos_weight.
#   scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
#   This tells XGBoost how much more weight to give the minority (churn) class.
#
# Step 2: Create xgb_model.
#   xgb_model = xgb.XGBClassifier(
#       scale_pos_weight=...,
#       n_estimators=100,
#       max_depth=4,
#       learning_rate=0.1,
#       eval_metric='logloss',
#       random_state=42
#   )
#
# Step 3: Update the pipeline to use xgb_model as the classifier.
#   pipe_xgb.set_params(classifier=xgb_model)
#
# Step 4: Fit the pipeline on X_train, y_train.

# Your Code Here:


### 4.3 Out-of-Fold Probabilities (GIVEN)

`cross_val_predict` trains the model on 4 of 5 folds and predicts the held-out fold, cycling through all folds.  
The result is a full set of training-set predictions where **each sample was predicted by a model that never saw it**.  

This lets us tune thresholds on training data without ever touching the test set.

You will use `oof_proba` in Section 7.


In [12]:
# GIVEN — uncomment and run this cell AFTER completing the TASKs above.
#
# print('Generating OOF probabilities (may take a moment)...')
# oof_proba = cross_val_predict(
#     pipe_xgb, X_train, y_train, cv=5, method='predict_proba'
# )[:, 1]
# print('Done. OOF proba shape:', oof_proba.shape)


## Section 5: Evaluate the Baseline Model

### 5.1 Business Utility vs. Naive Strategies — TASK

Before looking at accuracy, compare the model's *business cost* against two naive strategies:

- **"Call no one":** Predict no churn for every customer. You pay \$0 in FP costs but lose \$200 for every actual churner.
- **"Call everyone":** Predict churn for every customer. You retain every churner but pay \$25 for every non-churner.

Use `COST_FN` and `COST_FP` in your calculation.


In [13]:
# TASK: Compare the model's cost against two naive strategies on the TEST SET.
#
# Step 1: Get predictions.
#   y_pred = pipe_xgb.predict(X_test)
#
# Step 2: Unpack the confusion matrix.
#   tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
#
# Step 3: Compute costs.
#   cost_call_none  = (number of actual positives in test set) * COST_FN
#   cost_call_all   = (number of actual negatives in test set) * COST_FP
#   cost_model      = fn * COST_FN + fp * COST_FP
#
# Step 4: Print all three costs with clear labels.

# Your Code Here:


**Interpretation:** *(Write your answer here.)*

Does the model beat both naive strategies? By how much?


### 5.2 Confusion Matrix — TASK


In [14]:
# TASK: Plot the confusion matrix.
#
# 1. Use ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
#    Set display_labels=['No Churn', 'Churn']
#
# 2. After the plot, print the four confusion matrix values with business labels:
#    - FP: 'Unnecessary retention offers sent'
#    - FN: 'Missed churners (revenue lost)'

# Your Code Here:


### 5.3 Classification Metrics — TASK


In [15]:
# TASK: Print precision, recall, and F1 for the positive (Churn) class.
# Then print the full classification_report with target_names=['No Churn', 'Churn'].

# Your Code Here:


**Interpretation:** *(Write your answer here.)*

In plain English, what do precision and recall mean for this churn prediction task?  
Which error type (FP or FN) is more costly, and why does that matter for how we should evaluate this model?


## Section 6: ROC and PR Curves

### 6.1 ROC Curve — TASK

The ROC curve shows the trade-off between True Positive Rate (recall) and False Positive Rate at every possible threshold.  
AUC = 1.0 is perfect; AUC = 0.5 is a random classifier.

**Grading scale from notebook 4:**  
`0.50` = no skill | `0.60–0.70` = poor | `0.70–0.80` = acceptable | `0.80–0.90` = excellent | `0.90+` = outstanding


In [16]:
# TASK: Plot the ROC curve and compute AUC.
#
# Step 1: Get test probabilities.
#   y_proba = pipe_xgb.predict_proba(X_test)[:, 1]
#
# Step 2: Compute ROC values.
#   fpr, tpr, _ = roc_curve(y_test, y_proba)
#   roc_auc = roc_auc_score(y_test, y_proba)
#
# Step 3: Plot.
#   - Plot fpr vs. tpr with AUC in the legend label
#   - Add the random-classifier diagonal (dashed line from (0,0) to (1,1))
#   - Label axes, add title and legend
#
# Step 4: Print the AUC and interpret it using the grading scale above.

# Your Code Here:


### 6.2 Precision-Recall Curve — TASK

The PR curve is especially informative when classes are imbalanced.  
The **baseline** for a random classifier is the positive rate (\~26.5% for this dataset).


In [17]:
# TASK: Plot the Precision-Recall curve and compute Average Precision.
#
# Step 1: Compute PR values.
#   precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
#   avg_precision = average_precision_score(y_test, y_proba)
#
# Step 2: Plot.
#   - Plot recall_vals vs. precision_vals
#   - Add a horizontal dashed baseline at y = y_test.mean()
#     (this represents what random guessing achieves on this dataset)
#   - Label axes, add title and legend
#
# Step 3: Print the Average Precision (PR AUC).

# Your Code Here:


**Comparison:** *(Write your answer here.)*

The Churn dataset has \~26.5% positive rate. The credit card fraud dataset (notebook 5/6) had only 0.17%.  
How do the ROC AUC and PR AUC compare for this dataset versus what we saw in fraud detection?  
Why is the gap between ROC AUC and PR AUC different at different levels of imbalance?


## Section 7: Threshold Tuning

The default 0.5 threshold assumes equal cost for FN and FP.  
With `COST_FN = $200` and `COST_FP = $25`, the break-even point is much lower:  
we should flag a customer as likely-to-churn as long as the probability exceeds roughly `COST_FP / (COST_FN + COST_FP)`.

We tune thresholds on **OOF training probabilities** (generated in Section 4.3) so the test set remains unseen.


### 7.1 Youden's J Threshold — TASK

Youden's J statistic maximizes the sum of sensitivity and specificity:  
**J = TPR − FPR** (maximize this over all candidate thresholds).

This is the threshold that maximizes overall discrimination ability, without regard to cost asymmetry.


In [18]:
# TASK: Find the Youden's J optimal threshold using OOF training probabilities.
#
# Step 1: Compute the ROC curve on the training OOF probabilities.
#   fpr_oof, tpr_oof, thresh_oof = roc_curve(y_train, oof_proba)
#
# Step 2: Compute Youden's J for each threshold.
#   youden_j = tpr_oof - fpr_oof
#
# Step 3: Find the best threshold.
#   best_thresh_youden = thresh_oof[np.argmax(youden_j)]
#   Print the threshold value.
#
# Step 4: Evaluate on the TEST SET at this threshold.
#   y_pred_youden = (y_proba >= best_thresh_youden).astype(int)
#   Print precision and recall for the Churn class.

# Your Code Here:


### 7.2 Cost-Optimal Threshold — TASK

Youden's J maximizes discrimination but ignores cost asymmetry.  
The cost-optimal threshold minimizes the total dollar cost on the OOF training set.


In [19]:
# TASK: Find the cost-optimal threshold by sweeping over candidate thresholds.
#
# Algorithm:
# Step 1: Define candidate thresholds.
#   candidate_thresholds = np.linspace(0.01, 0.99, 200)
#
# Step 2: For each threshold t, compute the OOF cost.
#   oof_costs = []
#   for t in candidate_thresholds:
#       yh = (oof_proba >= t).astype(int)
#       tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_train, yh).ravel()
#       oof_costs.append(fn_t * COST_FN + fp_t * COST_FP)
#
# Step 3: Find the best threshold.
#   best_thresh_cost = candidate_thresholds[np.argmin(oof_costs)]
#   Print the threshold value.
#
# Step 4: Plot the cost curve.
#   - x = candidate_thresholds, y = oof_costs
#   - Add vertical dashed lines for: Default (0.50), Youden (best_thresh_youden),
#     Cost-Optimal (best_thresh_cost)
#   - Label axes and add a legend

# Your Code Here:


### 7.3 Threshold Comparison Table — TASK

The helper function `eval_thresh` below evaluates any threshold on the test set and returns a summary dict.  
Call it for each of the three strategies and display the results.


In [20]:
def eval_thresh(y_true, y_proba, thresh, label):
    y_p = (y_proba >= thresh).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_true, y_p).ravel()
    prec = precision_score(y_true, y_p, zero_division=0)
    rec  = recall_score(y_true, y_p)
    cost = fn_t * COST_FN + fp_t * COST_FP
    return {
        'Strategy':    label,
        'Threshold':   f'{thresh:.4f}',
        'Precision':   f'{prec:.3f}',
        'Recall':      f'{rec:.3f}',
        'Test Cost ($)': f'{cost:,.0f}',
        '_cost':       cost,
    }


In [21]:
# TASK: Call eval_thresh for each strategy and display the comparison table.
#
# Call it three times:
#   eval_thresh(y_test, y_proba, 0.50,               'Default (0.50)')
#   eval_thresh(y_test, y_proba, best_thresh_youden,  'Youden\'s J')
#   eval_thresh(y_test, y_proba, best_thresh_cost,    'Cost-Optimal')
#
# Collect the results in a list, build a DataFrame, and display it.
#
# Then compute and print the dollar savings of the best strategy
# vs. the default threshold.

# Your Code Here:


## Section 8: Nested Cross-Validation and the Final Model

### 8.1 Why Nested CV?

Whenever we tune hyperparameters (via `GridSearchCV`), the outer CV score is optimistically biased — we picked the best params *for* that outer fold.  
**Nested CV** corrects this: the inner loop tunes params, the outer loop evaluates the tuned model on held-out data it never influenced.  

This gives us the most honest estimate of how well this approach generalizes to new customers.

For a full walkthrough of nested CV, see notebook 6.


### 8.2 Run Nested CV — TASK (fill in the blanks)

Complete the `param_grid` below and run the nested CV.


In [22]:
# GIVEN — uncomment and run this cell AFTER completing the TASKs above.
# (Remember to fill in the three ... placeholders in param_grid first.)
#
# TASK: Fill in the three ... placeholders in the param_grid.
#
# Use these search values:
#   n_estimators:  [50, 100]
#   max_depth:     [3, 5]
#   learning_rate: [0.05, 0.1]
#
# Note: the 'classifier__' prefix is required by sklearn Pipeline syntax.

# param_grid = {
#     'classifier__n_estimators':  ...,
#     'classifier__max_depth':     ...,
#     'classifier__learning_rate': ...,
# }

# inner_grid = GridSearchCV(
#     pipe_xgb, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0
# )
# outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# print('Running nested CV (this may take a few minutes)...')
# nested_scores = cross_val_score(
#     inner_grid, X, y, cv=outer_cv, scoring='roc_auc', n_jobs=-1
# )

# print('Nested CV AUC scores:', np.round(nested_scores, 4))
# print('Mean: {:.4f}  +/-  {:.4f}'.format(nested_scores.mean(), nested_scores.std()))


**Interpretation:** *(Write your answer here.)*

How does the nested CV mean AUC compare to the single test-set AUC from Section 6?  
What would wide variation across the 5 folds tell you about this model?


### 8.3 Final Production Model (GIVEN)

To build the final model, we tune hyperparameters on the **full dataset** and train the winner.  
We do **not** evaluate this model on any data — we already have our honest generalization estimate from nested CV.


In [23]:
# GIVEN — uncomment and run this cell AFTER completing the TASKs above.
#
# GIVEN: Train the final production model on all available data.
# Do NOT call .score() or predict() on X after this cell.

# final_grid = GridSearchCV(
#     pipe_xgb, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
# )
# final_grid.fit(X, y)

# print('Best params:', final_grid.best_params_)
# print('Best inner CV AUC: {:.4f}'.format(final_grid.best_score_))
# print()
# print('Recommended production threshold: {:.4f}'.format(best_thresh_cost))
# print('(This threshold was selected on OOF training probabilities in Section 7)')


## Section 9: Discussion Questions

Answer each question in complete sentences. Use specific numbers from your results where relevant.


### Question 1 — Accuracy Paradox

The naive baseline for this dataset is \~73.5%. Your XGBoost model almost certainly achieved higher accuracy.  
Does higher accuracy automatically make it a better model than the naive strategy?  
Explain using the concepts of precision, recall, and business cost.


**Your Answer:** *(Write your answer here.)*


### Question 2 — Class Imbalance Strategy

You chose a specific class imbalance strategy in Section 4. Looking back, was it the right choice?  

Compare the class ratio and cost ratio for Telco Churn to the same numbers in the credit card fraud notebook (notebook 6).  
What would have been wrong with applying the custom objective approach used in fraud detection to this dataset?


**Your Answer:** *(Write your answer here.)*


### Question 3 — Threshold Selection

The cost-optimal threshold should be substantially below 0.5. Explain in business terms why this makes sense.  

What operational constraint might push you to use a *higher* threshold than the mathematical optimum,  
even if it increases total cost?


**Your Answer:** *(Write your answer here.)*


### Question 4 — ROC vs. PR AUC

This dataset has \~26.5% positive rate. Notebook 5 (credit card fraud) had 0.17%.  

Compare how "honest" ROC AUC is as an evaluation metric in each case.  
Why does the severity of class imbalance change the relationship between ROC AUC and PR AUC?


**Your Answer:** *(Write your answer here.)*


### Question 5 — Nested CV Stability

Your nested CV produces 5 AUC scores. Suppose they varied widely — for example: 0.71, 0.79, 0.72, 0.80, 0.73.  

What would this instability mean? How would you communicate it to a non-technical business stakeholder who only wants one number?


**Your Answer:** *(Write your answer here.)*


### Question 6 — Feature Importance and Data Leakage

Plot the feature importances of your final XGBoost model:  
```python
importances = final_grid.best_estimator_.named_steps['classifier'].feature_importances_
# The preprocessor's get_feature_names_out() can give you transformed column names.
```

Which features are most predictive? Do any of them raise **data leakage** concerns?  
*(Think carefully: could any feature only be known *after* a customer has already churned?)*


**Your Answer:** *(Write your answer here.)*
